In [1]:
import json
import glob
import os
from eval_utils import extract_decision, get_eval_accuracy
from datasets import load_dataset

def parse_checkpoint(f):
    # Assuming format like "prefix_ckpt{idx}_suffix"
    base = f.split('ckpt')[0].strip('_')  # Get the prefix before 'ckpt'
    remaining = f.split('ckpt')[1]
    
    # Split the remaining part into idx and suffix
    idx = int(remaining.split('_')[0])  # Extract the number after 'ckpt'
    suffix = remaining.split('_')[1] if '_' in remaining else ''  # Get suffix if exists
    suffix = suffix.strip('.json')
    return base, idx, suffix

# glob all files in pub-med-eval
subset = "labeled"
files = glob.glob(f"pub-med-eval-v2/cpt/*_subset={subset}.json")
true_data = load_dataset("qiaojin/PubMedQA", f"pqa_{subset}", split="train")


/opt/poetry-venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def evaluate_pubmed_predictions(file_path, true_data):
    """
    Evaluate model predictions against ground truth for PubMedQA dataset.
    
    Args:
        file_path: Path to the JSON file containing model generations
        true_data: Dataset containing ground truth decisions
        
    Returns:
        float: Accuracy of the predictions
    """
    # Load predictions
    pred_data = json.load(open(file_path, 'r'))
    generations = pred_data["generations_str"]
    
    # Extract decisions from generations
    def extract_decision_text(text):
        if not text:
            return None

        text = text.rstrip('#').lower().strip('.').strip('\n')
        if text.endswith('yes'):
            return 'yes'
        elif text.endswith('no'):
            return 'no'
        elif text.endswith('maybe'):
            return 'maybe'
        
        if 'no.' in text:
            return 'no'
        elif 'yes.' in text:
            return 'yes'
        elif 'maybe.' in text:
            return 'maybe'

        try:
            if (s := 'final decision:') in text:
                decision = text.split(s)[1]
            elif (s := 'final decision is') in text:
                decision = text.split(s)[1]
            else:
                decision = text
            decision = decision.strip()          
            # remove quotes
            decision = decision.strip('"').strip("'")
            return decision.strip('.:').strip()
        except Exception as e:
            print(f"Error: {e}\n{text}\n")
            return None
    
    # Process all generations
    pred_decisions = []
    extraction_failures = 0
    for generation in generations:
        completion = generation.split('assistant\n\n')[1]
        decision = extract_decision_text(completion)
        if decision is None or decision not in ['yes', 'no', 'maybe']:
            # print(f"Error extracting decision")
            # print("Extracted Decision-", decision)
            extraction_failures += 1
        pred_decisions.append(decision)
    
    # Get ground truth decisions
    true_decisions = true_data["final_decision"]
    
    # Calculate and return accuracy
    accuracy = get_eval_accuracy(pred_decisions, true_decisions)
    print(f"Extraction failure count: {extraction_failures}")
    return accuracy

# Evaluate the first file
for f in files:
    accuracy = evaluate_pubmed_predictions(f, true_data)
    print(f)
    print(f"Accuracy: {accuracy:.4f}")


Extraction failure count: 29
pub-med-eval-v2/cpt/mora_cpt_witheval_ckpt3200_subset=labeled.json
Accuracy: 0.4460
Extraction failure count: 16
pub-med-eval-v2/cpt/mora_cpt_witheval_ckpt3200_rerun_subset=labeled.json
Accuracy: 0.4680
Extraction failure count: 22
pub-med-eval-v2/cpt/mora_cpt_witheval_ckpt6400_subset=labeled.json
Accuracy: 0.4730
Extraction failure count: 17
pub-med-eval-v2/cpt/mora_cpt_witheval_ckpt9600_subset=labeled.json
Accuracy: 0.4580
